# Embedding Generation


1. `all-MiniLM-L6-v2` 
2. `nomic-ai/nomic-embed-text-v1` 
3. `text-embedding-3-small` 


In [16]:
!pip install sentence-transformers openai numpy einops

In [1]:
import json
import os
import time
import numpy as np
from typing import List, Dict, Any

In [ ]:
# Config
INPUT_PATH = "../chuncking/data/chunks_fixed300_overlap20pct.jsonl" 
OUTPUT_DIR = "./output/chunks_fixed300_overlap20"
BATCH_SIZE = 16

## Load Chunks

In [26]:
def load_chunks(path: str) -> List[Dict[str, Any]]:
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    print(f"Loaded {len(chunks)} chunks from {path}")
    return chunks

chunks = load_chunks(INPUT_PATH)
texts = [c["text"] for c in chunks]

Loaded 2253 chunks from ../chuncking/data/chunks_fixed300_overlap20pct.jsonl


In [27]:
def save_embedded_chunks(
    chunks: List[Dict],
    embeddings: np.ndarray,
    output_path: str,
):
    """Write chunks with embeddings filled in to JSONL."""
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for chunk, emb in zip(chunks, embeddings):
            record = {
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"],
                "embedding": emb.tolist(),
                "metadata": chunk["metadata"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Saved to {output_path} ({size_mb:.2f} MB)")

## Model 1: all-MiniLM-L6-v2

Lightweight sentence-transformer (22M params, 384d). Fast and widely used as a baseline.

In [28]:
from sentence_transformers import SentenceTransformer

model_minilm = SentenceTransformer("all-MiniLM-L6-v2")

t0 = time.time()
embeddings_minilm = model_minilm.encode(
    texts, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True
)
elapsed = time.time() - t0

print(f"Shape: {embeddings_minilm.shape}")
print(f"Time : {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

save_embedded_chunks(
    chunks, embeddings_minilm,
    os.path.join(OUTPUT_DIR, "embedded_minilm.jsonl")
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 696.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 141/141 [00:25<00:00,  5.59it/s]


Shape: (2253, 384)
Time : 25.4s (89 chunks/s)
Saved to ./output/chunks_fixed300_overlap20\embedded_minilm.jsonl (21.16 MB)


## Model 2: nomic-ai/nomic-embed-text-v1

Open-source 768d model with strong MTEB scores. Requires a task prefix for documents.

In [29]:
model_nomic = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1", trust_remote_code=True
)

# nomic requires prefix: "search_document: " for indexing, "search_query: " for queries
texts_prefixed = [f"search_document: {t}" for t in texts]

t0 = time.time()
embeddings_nomic = model_nomic.encode(
    texts_prefixed, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True
)
elapsed = time.time() - t0

print(f"Shape: {embeddings_nomic.shape}")
print(f"Time : {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

save_embedded_chunks(
    chunks, embeddings_nomic,
    os.path.join(OUTPUT_DIR, "embedded_nomic.jsonl")
)

<All keys matched successfully>
Batches: 100%|██████████| 141/141 [04:55<00:00,  2.10s/it]


Shape: (2253, 768)
Time : 296.1s (8 chunks/s)
Saved to ./output/chunks_fixed300_overlap20\embedded_nomic.jsonl (39.51 MB)


## Model 3: text-embedding-3-small (OpenAI API)

OpenAI's small embedding model (1536d). Requires an API key. Batched to respect rate limits.

In [30]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("Enter OpenAI API key: ")

client = OpenAI(api_key=OPENAI_API_KEY)
OPENAI_MODEL = "text-embedding-3-small"
OPENAI_BATCH = 100 


def embed_openai(texts: List[str], model: str = OPENAI_MODEL, batch_size: int = OPENAI_BATCH) -> np.ndarray:
    """Call OpenAI embedding API in batches."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(input=batch, model=model)
        batch_embs = [item.embedding for item in resp.data]
        all_embs.extend(batch_embs)
        if i + batch_size < len(texts):
            time.sleep(0.5)  # basic rate-limit backoff
    return np.array(all_embs, dtype=np.float32)


t0 = time.time()
embeddings_openai = embed_openai(texts)
elapsed = time.time() - t0

print(f"Shape: {embeddings_openai.shape}")
print(f"Time : {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

save_embedded_chunks(
    chunks, embeddings_openai,
    os.path.join(OUTPUT_DIR, "embedded_openai_small.jsonl")
)

Shape: (2253, 1536)
Time : 39.2s (57 chunks/s)
Saved to ./output/chunks_fixed300_overlap20\embedded_openai_small.jsonl (76.51 MB)


## Summary

In [31]:
print(f"{'Model':35s} {'Dim':>6s} {'File':>30s}")
print("-" * 75)
for name, emb, fname in [
    ("all-MiniLM-L6-v2", embeddings_minilm, "embedded_minilm.jsonl"),
    ("nomic-embed-text-v1", embeddings_nomic, "embedded_nomic.jsonl"),
    ("text-embedding-3-small", embeddings_openai, "embedded_openai_small.jsonl"),
]:
    path = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"{name:35s} {emb.shape[1]:>6d} {fname:>30s} ({size:.1f} MB)")

Model                                  Dim                           File
---------------------------------------------------------------------------
all-MiniLM-L6-v2                       384          embedded_minilm.jsonl (21.2 MB)
nomic-embed-text-v1                    768           embedded_nomic.jsonl (39.5 MB)
text-embedding-3-small                1536    embedded_openai_small.jsonl (76.5 MB)


In [32]:
# Verify: load first record from each file
for fname in ["embedded_minilm.jsonl", "embedded_nomic.jsonl", "embedded_openai_small.jsonl"]:
    path = os.path.join(OUTPUT_DIR, fname)
    with open(path, "r", encoding="utf-8") as f:
        rec = json.loads(f.readline())
    emb_len = len(rec["embedding"])
    print(f"[{fname}] chunk_id={rec['chunk_id']}  embedding_dim={emb_len}")

[embedded_minilm.jsonl] chunk_id=wiki_0001_chunk_0  embedding_dim=384
[embedded_nomic.jsonl] chunk_id=wiki_0001_chunk_0  embedding_dim=768
[embedded_openai_small.jsonl] chunk_id=wiki_0001_chunk_0  embedding_dim=1536


In [7]:

import faiss
import numpy as np
import json
import os

EMBEDDED_FILES = [
    "output/chunks_fixed300_overlap20/embedded_minilm.jsonl",
    "output/chunks_fixed300_overlap20/embedded_nomic.jsonl",
    "output/chunks_fixed300_overlap20/embedded_openai_small.jsonl",
]

INDEX_DIR = "output/chunks_fixed300_overlap20/"

In [8]:

def build_faiss_index(embedded_path: str, output_dir: str):
    """Load embedded JSONL, build and save FAISS index."""
    basename = os.path.splitext(os.path.basename(embedded_path))[0]

    embeddings_list = []
    with open(embedded_path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line.strip())
            if not rec.get("embedding"):
                continue
            embeddings_list.append(rec["embedding"])

    if not embeddings_list:
        print(f"[SKIP] No valid embeddings in {embedded_path}")
        return

    embeddings = np.array(embeddings_list, dtype=np.float32)
    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    os.makedirs(output_dir, exist_ok=True)
    index_path = os.path.join(output_dir, f"{basename}.faiss")
    faiss.write_index(index, index_path)

    print(f"[OK] {basename}: {index.ntotal} vectors, dim={embeddings.shape[1]} -> {index_path}")


for path in EMBEDDED_FILES:
    if not os.path.exists(path):
        print(f"[SKIP] Not found: {path}")
        continue
    build_faiss_index(path, INDEX_DIR)

[OK] embedded_minilm: 2253 vectors, dim=384 -> output/chunks_fixed300_overlap20/embedded_minilm.faiss
[OK] embedded_nomic: 2253 vectors, dim=768 -> output/chunks_fixed300_overlap20/embedded_nomic.faiss
[OK] embedded_openai_small: 2253 vectors, dim=1536 -> output/chunks_fixed300_overlap20/embedded_openai_small.faiss


In [ ]:
# Verify: load each index, query with first vector, look up text from original JSONL
for path in EMBEDDED_FILES:
    basename = os.path.splitext(os.path.basename(path))[0]
    index_path = os.path.join(INDEX_DIR, f"{basename}.faiss")
    if not os.path.exists(index_path):
        continue

    index = faiss.read_index(index_path)
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # Use first chunk as query (should match itself at rank 1)
    first = json.loads(lines[0])
    q = np.array([first["embedding"]], dtype=np.float32)
    faiss.normalize_L2(q)
    scores, indices = index.search(q, k=3)

    print(f"\n--- {basename} ({index.ntotal} vectors) ---")
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        rec = json.loads(lines[idx])
        print(f"  #{rank+1} score={score:.4f}  {rec['chunk_id']}  [{rec['metadata']['title']}]")
        print(f"       {rec['text'][:80]}...")



--- embedded_minilm (2253 vectors) ---
  #1 score=1.0000  wiki_0001_chunk_0  [List of Asian cuisines]
       East Asian cuisine has evolved with common usage of oils, fats and sauces in the...
  #2 score=0.9053  wiki_0033_chunk_3  [List of Asian cuisines]
       , history, cooking techniques and lifestyle. Many Chinese traditional regional c...
  #3 score=0.8496  wiki_0001_chunk_3  [List of Asian cuisines]
       China. Possibly Shandong's greatest contribution to Chinese cuisine has been in ...

--- embedded_nomic (2253 vectors) ---
  #1 score=1.0000  wiki_0001_chunk_0  [List of Asian cuisines]
       East Asian cuisine has evolved with common usage of oils, fats and sauces in the...
  #2 score=0.9492  wiki_0033_chunk_3  [List of Asian cuisines]
       , history, cooking techniques and lifestyle. Many Chinese traditional regional c...
  #3 score=0.9074  wiki_0001_chunk_1  [List of Asian cuisines]
       , connoisseurs of Chinese cuisine have also sprouted in Eastern Europe and South.